# MLP


In [1]:
import numpy as np
import matplotlib.pyplot as plt
#from tensorflow.keras.datasets import mnist
import tensorflow_datasets as tfds
from PIL import Image, ImageOps, ImageFilter, ImageEnhance
import random
from tqdm import tqdm, trange

In [2]:
class ActivationFunction:
    def sigmoid(x):
        return 1 / (1 + np.exp(-x))

    def sigmoid_derivative(x):
        sig = ActivationFunction.sigmoid(x)
        return sig * (1 - sig)

    def tanh(x):
        return np.tanh(x)

    def tanh_derivative(x):
        return 1 - np.tanh(x) ** 2

    def relu(x):
        return np.maximum(0, x)

    def relu_derivative(x):
        return (x > 0).astype(float)

    def softmax(x):
      x = x - np.max(x, axis=1, keepdims=True)  # évite les valeurs très grandes
      exp_vals = np.exp(x)
      sum_exp = np.sum(exp_vals, axis=1, keepdims=True)
      sum_exp[sum_exp == 0] = 1e-8  # évite division par zéro
      return exp_vals / sum_exp

class LossFunction:
    def mse(y_true, y_pred):
        return np.mean((y_true - y_pred) ** 2)

    def mse_derivative(y_true, y_pred):
        return (y_pred - y_true) / y_true.shape[0]

    def cross_entropy(y_true, y_pred):
      eps = 1e-15
      y_pred = np.clip(y_pred, eps, 1 - eps)
      return -np.sum(y_true * np.log(y_pred)) / y_true.shape[0]

    def cross_entropy_derivative(y_true, y_pred):
        return (y_pred - y_true) / y_true.shape[0]

In [30]:
class Layer:
    def __init__(self, n_inputs, n_neurons, activation_function, activation_derivative):
        """
        n_inputs: Nombre d'entrées par neurone
        n_neurons: Nombre de neurones dans la couche
        activation_function: Fonction d'activation
        activation_derivative: Dérivée de la fonction d'activation
        """
        self.n_inputs = n_inputs
        self.n_neurons = n_neurons
        self.activation_function = activation_function
        self.activation_derivative = activation_derivative

        # Initialisation des poids et biais
        self.weights = np.random.randn(n_inputs, n_neurons) * np.sqrt(1 / n_inputs)
        self.biases = np.zeros((1, n_neurons))

        # Variables pour la propagation arrière
        self.output = None
        self.input = None
        self.d_weights = None
        self.d_biases = None

    def forward(self, input_data):
        """
        Propagation avant

        input_data: Les entrées de cette couche
        return: La sortie de la couche après activation
        """
        self.input = input_data
        z = np.dot(input_data, self.weights) + self.biases
        self.output = self.activation_function(z)
        return self.output

    def backward(self, d_output, learning_rate):
      """
      Mise à jour des poids et biais de cette couche en appliquant la rétropropagation.
      """
      # Règle de la chaine
      if self.activation_derivative is not None:
          d_activation = d_output * self.activation_derivative(self.output)
      else:
          d_activation = d_output  # pour softmax + cross-entropy

      # Calcul des gradients pour les poids et biais
      d_weights = np.dot(self.input.T, d_activation)
      d_biases = np.sum(d_activation, axis=0, keepdims=True)

      d_activ_new = np.dot(d_activation, self.weights.T)

      # Mise à jour des poids et biais
      self.weights -= learning_rate * d_weights
      self.biases -= learning_rate * d_biases

      # Propagation des gradients à la couche précédente
      return d_activ_new

In [23]:
class MLP:
    def __init__(self, layers):
        """
        layers: Une liste de tuples (n_inputs, n_neurons, activation_function, activation_derivative)
        Chaque tuple correspond à une couche. Le premier élément est le nombre d'entrées, le deuxième est le nombre de neurones dans cette couche, et les deux derniers éléments sont la fonction d'activation et sa dérivée.
        """
        self.layers = []
        for i in range(len(layers)):
            n_inputs, n_neurons, activation_function, activation_derivative = layers[i]
            if i == 0:
                # La première couche, les entrées viennent de l'extérieur (pas d'entrées avant)
                self.layers.append(Layer(n_inputs, n_neurons, activation_function, activation_derivative))
            else:
                # Pour les couches suivantes, les n_inputs sont égaux au nombre de neurones de la couche précédente
                self.layers.append(Layer(self.layers[i-1]., n_neurons, activation_function, activation_derivative))

    def forward(self, X):
        """
        Propagation avant à travers toutes les couches.

        X: Les données d'entrée (n_samples, n_features)
        return: La sortie du modèle (n_samples, n_neurons_output)
        """
        input_data = X
        for layer in self.layers:
            input_data = layer.forward(input_data)
        return input_data

    def backward(self, X, y, loss_fn, learning_rate):
        """
        Propagation arrière à travers toutes les couches.

        X: Les données d'entrée
        y: Les étiquettes (vraies valeurs)
        loss_fn: La fonction de perte (ex: MSE)
        learning_rate: Le taux d'apprentissage
        """
        y_pred = self.forward(X)
        grad = loss_fn(y, y_pred)

        for layer in reversed(self.layers):
            grad = layer.backward(grad, learning_rate)

    def fit(self, X, y, epochs, batch_size, loss_fn, loss_fn_derivative, learning_rate):
        n_samples = X.shape[0]

        for epoch in trange(epochs, desc="Entraînement", leave=True):
            #Mélange aléatoire des données à chaque époque
            indices = np.arange(n_samples)
            np.random.shuffle(indices)
            X, y = X[indices], y[indices]

            loss_total = 0
            for i in range(0, n_samples, batch_size):
                batch_X = X[i:i+batch_size]
                batch_y = y[i:i+batch_size]
                self.backward(batch_X, batch_y, loss_fn_derivative, learning_rate)
                loss_total += LossFunction.cross_entropy(batch_y, self.forward(batch_X))

            avg_loss = loss_total / (n_samples // batch_size)
            accuracy = self.compute_accuracy(y, self.forward(X))

            tqdm.write(f" Epoch {epoch+1}/{epochs} - Loss: {avg_loss:.4f} - Accuracy: {accuracy:.4f}")
        save(mlp, "mnist_"+str(epochs)+".txt")

    def predict(self, X):
        output = self.forward(X)
        return np.argmax(output, axis=1)

    def compute_accuracy(self, y_true, y_pred):
        if y_pred.shape[1] == 1:
            y_pred_labels = (y_pred > 0.5).astype(int)
            y_true_labels = y_true
        else:
            y_pred_labels = np.argmax(y_pred, axis=1)
            y_true_labels = np.argmax(y_true, axis=1)
        return np.mean(y_pred_labels == y_true_labels)

def save(model, filename):
    with open(filename, "w") as f:
        for i, layer in enumerate(model.layers):
            f.write(f"# Layer {i}\n")
            np.savetxt(f, layer.weights, delimiter=",", fmt="%.6f")
            f.write("##\n")
            np.savetxt(f, layer.biases, delimiter=",", fmt="%.6f")
            f.write("###\n")
    print(f"Paramètres sauvegardés dans : {filename}")

def load(model, filename):
    with open(filename, "r") as f:
        lines = f.readlines()

    i = 0
    layer_idx = 0
    while i < len(lines):
        if lines[i].startswith("# Layer"):
            i += 1
            weights = []
            while i < len(lines) and not lines[i].startswith("##"):
                weights.append([float(x) for x in lines[i].strip().split(",")])
                i += 1
            i += 1  # Skip "##"
            biases = []
            while i < len(lines) and not lines[i].startswith("###"):
                biases.append([float(x) for x in lines[i].strip().split(",")])
                i += 1
            i += 1  # Skip "###"
            model.layers[layer_idx].weights = np.array(weights)
            model.layers[layer_idx].biases = np.array(biases)
            layer_idx += 1

    print(f"Paramètres chargés depuis : {filename}")

# MNIST

In [19]:
emnist = tfds.load('emnist/digits', split="all", as_supervised=True)

Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Extraction completed...: 0 file [00:00, ? file/s]

Extraction completed...: 0 file [00:00, ? file/s]

Generating splits...:   0%|          | 0/2 [00:00<?, ? splits/s]

Generating train examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/emnist/digits/incomplete.DFFIT2_3.1.0/emnist-train.tfrecord*...:   0%|    …

Generating test examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/emnist/digits/incomplete.DFFIT2_3.1.0/emnist-test.tfrecord*...:   0%|     …

Dataset emnist downloaded and prepared to /root/tensorflow_datasets/emnist/digits/3.1.0. Subsequent calls will reuse this data.


In [20]:
def load_mnist():
    (X_train, y_train), (X_test, y_test) = mnist.load_data()
    X_train = X_train.reshape(-1, 28*28) / 255.0
    X_test = X_test.reshape(-1, 28*28) / 255.0
    y_train_encoded = np.eye(10)[y_train]
    y_test_encoded = np.eye(10)[y_test]
    return X_train, y_train_encoded, X_test, y_test_encoded, y_test

def load_emnist():
    X_emnist = []
    y_emnist = []

    for image, label in tfds.as_numpy(emnist):
        if isinstance(image, str):  # sécurité en cas de mauvaise lecture
            continue
        img = image.astype(np.float32) / 255.0
        X_emnist.append(img.flatten())
        y_emnist.append(label)

    X_emnist = np.array(X_emnist)
    y_emnist = np.array(y_emnist)
    y_emnist_encoded = np.eye(10)[y_emnist]

    return X_emnist, y_emnist_encoded

def predict_image(model, image_path):
    img = Image.open(image_path).convert("L").resize((28, 28))
    img2 = img

    # Amélioration de la robustesse : inverser si image claire
    if np.array(img).mean() > 127:
        img = ImageOps.invert(img)

    # Flou gaussien
    #img = img.filter(ImageFilter.GaussianBlur(radius=0.5))

    arr = np.array(img).astype(np.float32) / 255.0
    arr = arr.reshape(1, -1)

    prediction = np.argmax(model.predict(arr), axis=1)[0]
    print(f"Prédiction : {prediction}")
    plt.imshow(img2, cmap="gray")
    plt.title(f"Prédit : {prediction}")
    plt.axis('off')
    plt.show()

def augment_mnist_dataset(X, y, num_augments=1):
    X_augmented, y_augmented = [], []
    for i in tqdm(range(len(X)), desc="Augmenting dataset"):
        img = (X[i].reshape(28, 28) * 255).astype(np.uint8)
        pil_img = Image.fromarray(img)
        for _ in range(num_augments):
            aug = pil_img.copy()
            if random.random() < 0.5:
                aug = ImageOps.invert(aug)
            if random.random() < 0.3:
                aug = aug.filter(ImageFilter.GaussianBlur(radius=0.5))
            if random.random() < 0.3:
                shift = random.randint(-2, 2)
                aug = ImageOps.expand(aug, border=(shift, shift, 0, 0), fill=0)
                aug = aug.crop((0, 0, 28, 28))
            if random.random() < 0.3:
                enhancer = ImageEnhance.Contrast(aug)
                aug = enhancer.enhance(random.uniform(0.7, 1.3))
            if random.random() < 0.2:
                noise = np.array(aug).astype(np.float32)
                noise += np.random.normal(0, 10, noise.shape)
                noise = np.clip(noise, 0, 255).astype(np.uint8)
                aug = Image.fromarray(noise)
            arr = np.array(aug).astype(np.float32) / 255.0
            X_augmented.append(arr.flatten())
            y_augmented.append(y[i])
    X_all = np.concatenate([X, np.array(X_augmented)], axis=0)
    y_all = np.concatenate([y, np.array(y_augmented)], axis=0)
    return X_all, y_all

In [21]:
X_train,y_train = load_emnist()

In [28]:
# Chargement des données
X_aug, y_aug = augment_mnist_dataset(X_train, y_train, num_augments=1)

Augmenting dataset: 100%|██████████| 280000/280000 [00:37<00:00, 7391.40it/s]


In [ ]:
layers = [
    (784, 512, ActivationFunction.relu, ActivationFunction.relu_derivative),
    (512, 256, ActivationFunction.relu, ActivationFunction.relu_derivative),
    (256, 128, ActivationFunction.relu, ActivationFunction.relu_derivative),
    (128, 10, ActivationFunction.softmax, None)
]

mlp = MLP(layers)

# Entraînement
training = True

if training:
  mlp.fit(
    X_aug, y_aug,
    epochs=150,
    batch_size=128,
    loss_fn=LossFunction.cross_entropy,
    loss_fn_derivative=LossFunction.cross_entropy_derivative,
    learning_rate=0.001
  )
else:
  load(mlp, "3_mnist_31.txt")

Entraînement:   0%|          | 0/150 [00:00<?, ?it/s]

In [ ]:
# Prédictions
y_pred = mlp.predict(X_aug[:10000])

NameError: name 'y' is not defined

In [ ]:
load(mlp, "3_mnist_31.txt")

predict_image(mlp, "/content/s_0.png")
predict_image(mlp, "/content/s_1.png")
predict_image(mlp, "/content/s_2.png")
predict_image(mlp, "/content/s_3.png")
predict_image(mlp, "/content/s_4.png")
predict_image(mlp, "/content/s_5.png")
predict_image(mlp, "/content/s_6.png")
predict_image(mlp, "/content/s_7.png")
predict_image(mlp, "/content/s_8.png")
predict_image(mlp, "/content/s_9.png")

NameError: name 'load_mlp_parameters' is not defined